In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.5 MB/s eta 0:00:00


In [3]:
import json

BASE = "/content/drive/MyDrive/Just_Advisor_Ai/legal_kb"

with open(f"{BASE}/laws.json") as f:
    laws = json.load(f)

with open(f"{BASE}/cases.json") as f:
    cases = json.load(f)

print("Laws loaded:", len(laws))
print("Cases loaded:", len(cases))


Laws loaded: 25
Cases loaded: 20


In [4]:
documents = []
metadata = []

for law in laws:
    documents.append(law["title"] + ". " + law["text"])
    metadata.append({"type": "law", "id": law["id"], "title": law["title"]})

for case in cases:
    documents.append(case["title"] + ". " + case["summary"])
    metadata.append({"type": "case", "id": case["id"], "title": case["title"]})


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(documents)

embeddings.shape


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(45, 384)

In [6]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Vector DB size:", index.ntotal)


Vector DB size: 45


In [7]:
def retrieve_context(query, k=5):
    q_emb = embedder.encode([query])
    D, I = index.search(np.array(q_emb), k)

    results = []
    for idx in I[0]:
        results.append(metadata[idx])

    return results


In [8]:
def ai_assistant(case_summary, lawyer_arguments):
    query = case_summary + " " + " ".join(lawyer_arguments)

    retrieved = retrieve_context(query)

    suggestions = {
        "relevant_laws": [],
        "relevant_cases": []
    }

    for item in retrieved:
        if item["type"] == "law":
            suggestions["relevant_laws"].append(item)
        else:
            suggestions["relevant_cases"].append(item)

    return suggestions


In [9]:
case_summary = "The accused received money but failed to deliver goods."

arguments = [
    "Dishonest intention existed from the beginning",
    "Bank transfer proves payment"
]

result = ai_assistant(case_summary, arguments)
result


{'relevant_laws': [{'type': 'law',
   'id': 'IPC_420',
   'title': 'Cheating and dishonestly inducing delivery of property'},
  {'type': 'law', 'id': 'CONTRACT_17', 'title': 'Fraud'}],
 'relevant_cases': [{'type': 'case',
   'id': 'CASE_001',
   'title': 'State of Maharashtra vs Mohd. Yakub'},
  {'type': 'case',
   'id': 'CASE_009',
   'title': 'Vesa Holdings vs State of Kerala'},
  {'type': 'case',
   'id': 'CASE_010',
   'title': 'Alpic Finance Ltd vs P. Sadasivan'}]}

In [10]:
!pip install transformers accelerate torch


In [17]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import torch

LLM_NAME = "microsoft/phi-2"

# Load the tokenizer
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

# Ensure pad_token is set for the tokenizer
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

# Load the model configuration
config = AutoConfig.from_pretrained(LLM_NAME)

# Set the pad_token_id in the configuration BEFORE loading the model
if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
    config.pad_token_id = llm_tokenizer.eos_token_id

# Load the model with the modified configuration
llm_model = AutoModelForCausalLM.from_pretrained(LLM_NAME, config=config)

# It's also good practice to ensure the generation_config is aligned
llm_model.generation_config.pad_token_id = config.pad_token_id

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [18]:
def generate_explanation(case_summary, laws, cases):
    laws_text = ", ".join([l["title"] for l in laws[:2]])
    cases_text = ", ".join([c["title"] for c in cases[:2]])

    prompt = f"""
Case: {case_summary}

Relevant laws: {laws_text}
Relevant cases: {cases_text}

Explain briefly (2 lines) why these laws and cases apply.
"""

    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True)

    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.3
        )

    text = llm_tokenizer.decode(output[0], skip_special_tokens=True)

    return text.split("Explain briefly")[-1].strip()


In [19]:
def ai_assistant(case_summary, lawyer_arguments):
    query = case_summary + " " + " ".join(lawyer_arguments)
    retrieved = retrieve_context(query)

    laws = []
    cases_ = []

    for item in retrieved:
        if item["type"] == "law":
            laws.append(item)
        else:
            cases_.append(item)

    explanation = generate_explanation(case_summary, laws, cases_)

    return {
        "relevant_laws": laws,
        "relevant_cases": cases_,
        "explanation": explanation
    }

In [20]:
case_summary = "The accused took money but failed to deliver goods."

arguments = ["Dishonest intention existed from start"]

result = ai_assistant(case_summary, arguments)

print(result["explanation"])



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


(2 lines) why these laws and cases apply.

Solution:

The accused took money but failed to deliver goods, which is a form of cheating and dishonestly inducing delivery of property. This is a violation of the law of fraud, which prohibits any act of deceit or misrepresentation to obtain property or money from another person. The accused is
